<table class="table table-bordered">
    <tr>
        <th style="text-align:center;"><h1>Data Wrangling</h1><h2>Assignment</h2><h3>Diploma in Data Science</h3></th>
    </tr>
</table>

**Prediction problem:** predict whether a Major League Baseball **position player** (a batter) is
selected for the **All-Star Game** in a given season, using his current-season batting statistics,
personal profile and team performance.

In [1]:
# import the packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_columns', 40)

### Step 1: Load and Explore the Data

In [2]:
# load the 4 tables needed for this problem
DATA_PATH = 'csv'

batting  = pd.read_csv(DATA_PATH + '/batting.csv')
player   = pd.read_csv(DATA_PATH + '/player.csv')
team     = pd.read_csv(DATA_PATH + '/team.csv')
all_star = pd.read_csv(DATA_PATH + '/all_star.csv')

batting.shape, player.shape, team.shape, all_star.shape

((101332, 22), (18846, 24), (2805, 48), (5069, 8))

In [3]:
# inspect the first rows of each table
batting.head()

,player_id,year,stint,team_id,league_id,g,ab,r,h,double,triple,hr,rbi,sb,cs,bb,so,ibb,hbp,sh,sf,g_idp
0,abercda01,1871,1,TRO,NaN,1,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,addybo01,1871,1,RC1,NaN,25,118.0,30.0,32.0,6.0,0.0,0.0,13.0,8.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN
2,allisar01,1871,1,CL1,NaN,29,137.0,28.0,40.0,4.0,5.0,0.0,19.0,3.0,1.0,2.0,5.0,NaN,NaN,NaN,NaN,NaN
3,allisdo01,1871,1,WS3,NaN,27,133.0,28.0,44.0,10.0,2.0,2.0,27.0,1.0,1.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN
4,ansonca01,1871,1,RC1,NaN,25,120.0,29.0,39.0,11.0,3.0,0.0,16.0,6.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN


In [4]:
player.head()

,player_id,birth_year,birth_month,birth_day,birth_country,birth_state,birth_city,death_year,death_month,death_day,death_country,death_state,death_city,name_first,name_last,name_given,weight,height,bats,throws,debut,final_game,retro_id,bbref_id
0,aardsda01,1981.0,12.0,27.0,USA,CO,Denver,NaN,NaN,NaN,NaN,NaN,NaN,David,Aardsma,David Allan,220.0,75.0,R,R,2004-04-06,2015-08-23,aardd001,aardsda01
1,aaronha01,1934.0,2.0,5.0,USA,AL,Mobile,NaN,NaN,NaN,NaN,NaN,NaN,Hank,Aaron,Henry Louis,180.0,72.0,R,R,1954-04-13,1976-10-03,aaroh101,aaronha01
2,aaronto01,1939.0,8.0,5.0,USA,AL,Mobile,1984.0,8.0,16.0,USA,GA,Atlanta,Tommie,Aaron,Tommie Lee,190.0,75.0,R,R,1962-04-10,1971-09-26,aarot101,aaronto01
3,aasedo01,1954.0,9.0,8.0,USA,CA,Orange,NaN,NaN,NaN,NaN,NaN,NaN,Don,Aase,Donald William,190.0,75.0,R,R,1977-07-26,1990-10-03,aased001,aasedo01
4,abadan01,1972.0,8.0,25.0,USA,FL,Palm Beach,NaN,NaN,NaN,NaN,NaN,NaN,Andy,Abad,Fausto Andres,184.0,73.0,L,L,2001-09-10,2006-04-13,abada001,abadan01


In [5]:
team.head()

,year,league_id,team_id,franchise_id,div_id,rank,g,ghome,w,l,div_win,wc_win,lg_win,ws_win,r,ab,h,double,triple,hr,...,era,cg,sho,sv,ipouts,ha,hra,bba,soa,e,dp,fp,name,park,attendance,bpf,ppf,team_id_br,team_id_lahman45,team_id_retro
0,1871,NaN,BS1,BNA,NaN,3,31,NaN,20,10,NaN,NaN,N,NaN,401,1372,426,70,37,3,...,3.55,22,1,3,828,367,2,42,23,225,NaN,0.83,Boston Red Stockings,South End Grounds I,NaN,103,98,BOS,BS1,BS1
1,1871,NaN,CH1,CNA,NaN,2,28,NaN,19,9,NaN,NaN,N,NaN,302,1196,323,52,21,10,...,2.76,25,0,1,753,308,6,28,22,218,NaN,0.82,Chicago White Stockings,Union Base-Ball Grounds,NaN,104,102,CHI,CH1,CH1
2,1871,NaN,CL1,CFC,NaN,8,29,NaN,10,19,NaN,NaN,N,NaN,249,1186,328,35,40,7,...,4.11,23,0,0,762,346,13,53,34,223,NaN,0.81,Cleveland Forest Citys,National Association Grounds,NaN,96,100,CLE,CL1,CL1
3,1871,NaN,FW1,KEK,NaN,7,19,NaN,7,12,NaN,NaN,N,NaN,137,746,178,19,8,2,...,5.17,19,1,0,507,261,5,21,17,163,NaN,0.80,Fort Wayne Kekiongas,Hamilton Field,NaN,101,107,KEK,FW1,FW1
4,1871,NaN,NY2,NNA,NaN,5,33,NaN,16,17,NaN,NaN,N,NaN,302,1404,403,43,21,1,...,3.72,32,1,0,879,373,7,42,22,227,NaN,0.83,New York Mutuals,Union Grounds (Brooklyn),NaN,90,88,NYU,NY2,NY2


In [6]:
all_star.head()

,player_id,year,game_num,game_id,team_id,league_id,gp,starting_pos
0,gomezle01,1933,0,ALS193307060,NYA,AL,1.0,1.0
1,ferreri01,1933,0,ALS193307060,BOS,AL,1.0,2.0
2,gehrilo01,1933,0,ALS193307060,NYA,AL,1.0,3.0
3,gehrich01,1933,0,ALS193307060,DET,AL,1.0,4.0
4,dykesji01,1933,0,ALS193307060,CHA,AL,1.0,5.0


In [7]:
# check for missing data in the columns we plan to use
batting[['player_id', 'year', 'g', 'ab', 'r', 'h', 'double', 'triple',
        'hr', 'rbi', 'sb', 'bb', 'so']].isnull().mean()

player_id    0.000000
year         0.000000
g            0.000000
ab           0.050813
r            0.050813
h            0.050813
double       0.050813
triple       0.050813
hr           0.050813
rbi          0.054997
sb           0.063642
bb           0.050813
so           0.128163
dtype: float64

In [8]:
player[['player_id', 'birth_year', 'weight', 'height', 'bats', 'debut']].isnull().mean()

player_id     0.000000
birth_year    0.007588
weight        0.046217
height        0.042715
bats          0.063196
debut         0.010241
dtype: float64

In [9]:
# check the cardinality of the categorical variable we will use
player['bats'].unique()

array(['R', 'L', nan, 'B'], dtype=object)

**Problem formulated:** this is a *binary classification* problem. For every position player-season,
the target `all_star` is 1 if the player was selected for that year's All-Star Game, and 0 otherwise.
We use **4 different tables**, `batting`, `player`, `team` and `all_star`, which satisfies the
requirement to combine information from multiple tables. Player salary was deliberately **not**
used as the prediction target.

### Step 2: Data Wrangling on multiple tables

In [10]:
# batting has one row per player per STINT (a player traded mid-season has 2+ rows
# for that year), so we group by player and year and sum the counting stats

season_cols = ['g', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'sb', 'bb', 'so']

batting_season = batting.groupby(['player_id', 'year'])[season_cols].sum().reset_index()

batting_season.shape

(93898, 13)

In [11]:
# keep only player-seasons with at least 20 games - the All-Star game is played
# mid-season, so players with very few games realistically cannot be selected

batting_season = batting_season[batting_season['g'] >= 20]

batting_season.shape

(66229, 13)

In [12]:
# grab the team each player played for that season (first team on record,
# for players traded mid-season)

team_lookup = batting[['player_id', 'year', 'team_id']].drop_duplicates(subset=['player_id', 'year'])

batting_season = pd.merge(batting_season, team_lookup, on=['player_id', 'year'], how='left')

batting_season.head()

,player_id,year,g,ab,r,h,double,triple,hr,rbi,sb,bb,so,team_id
0,aardsda01,2006,45,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CHN
1,aardsda01,2007,25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CHA
2,aardsda01,2008,47,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,BOS
3,aardsda01,2009,73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SEA
4,aardsda01,2010,53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SEA


In [13]:
# mathematical calculation: batting average = hits / at-bats

batting_season['avg'] = batting_season['h'] / batting_season['ab']

batting_season[['player_id', 'year', 'ab', 'h', 'avg']].head()

,player_id,year,ab,h,avg
0,aardsda01,2006,2.0,0.0,0.0
1,aardsda01,2007,0.0,0.0,NaN
2,aardsda01,2008,1.0,0.0,0.0
3,aardsda01,2009,0.0,0.0,NaN
4,aardsda01,2010,0.0,0.0,NaN


In [14]:
# merge in the player profile table

player_cols = player[['player_id', 'birth_year', 'weight', 'height', 'bats', 'debut']]

data = pd.merge(batting_season, player_cols, on='player_id', how='left')

data.shape

(66229, 20)

In [15]:
# extract the debut YEAR from the text/date column 'debut'

data['debut'] = pd.to_datetime(data['debut'])
data['debut_year'] = data['debut'].dt.year

data[['player_id', 'debut', 'debut_year']].head()

,player_id,debut,debut_year
0,aardsda01,2004-04-06,2004
1,aardsda01,2004-04-06,2004
2,aardsda01,2004-04-06,2004
3,aardsda01,2004-04-06,2004
4,aardsda01,2004-04-06,2004


In [16]:
# mathematical calculations: age and years of experience in that season

data['age'] = data['year'] - data['birth_year']
data['experience'] = data['year'] - data['debut_year']

data[['player_id', 'year', 'birth_year', 'age', 'debut_year', 'experience']].head()

,player_id,year,birth_year,age,debut_year,experience
0,aardsda01,2006,1981.0,25.0,2004,2
1,aardsda01,2007,1981.0,26.0,2004,3
2,aardsda01,2008,1981.0,27.0,2004,4
3,aardsda01,2009,1981.0,28.0,2004,5
4,aardsda01,2010,1981.0,29.0,2004,6


In [17]:
# merge in the team's win percentage for that season

team_cols = team[['year', 'team_id', 'w', 'g']].rename(columns={'g': 'team_g'})
team_cols['team_win_pct'] = team_cols['w'] / team_cols['team_g']

data = pd.merge(data, team_cols[['year', 'team_id', 'team_win_pct']],
                on=['year', 'team_id'], how='left')

data.shape

(66229, 24)

In [18]:
# build the target variable: was the player an All-Star that season?

all_star_flag = all_star[['player_id', 'year']].drop_duplicates()
all_star_flag['all_star'] = 1

data = pd.merge(data, all_star_flag, on=['player_id', 'year'], how='left')
data['all_star'] = data['all_star'].fillna(0)

data['all_star'].value_counts(normalize=True)

all_star
0.0    0.926679
1.0    0.073321
Name: proportion, dtype: float64

Only around 1 in 10 player-seasons is an All-Star selection, so the classes are imbalanced.
This means plain accuracy can be misleading, and we compare against a naive baseline in Step 4.

### Step 3: Data Cleansing and Transformation

In [19]:
# a quick look at where we still have missing values

data.isnull().mean()

player_id       0.000000
year            0.000000
g               0.000000
ab              0.000000
r               0.000000
h               0.000000
double          0.000000
triple          0.000000
hr              0.000000
rbi             0.000000
sb              0.000000
bb              0.000000
so              0.000000
team_id         0.000000
avg             0.089779
birth_year      0.000362
weight          0.004681
height          0.004590
bats            0.008531
debut           0.000000
debut_year      0.000000
age             0.000362
experience      0.000000
team_win_pct    0.000000
all_star        0.000000
dtype: float64

In [20]:
# drop the columns we will not use as model features (identifiers and raw dates)

features = ['g', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'sb', 'bb', 'so',
           'avg', 'weight', 'height', 'bats', 'age', 'experience', 'team_win_pct']

X = data[features]
y = data['all_star']

X.shape, y.shape

((66229, 18), (66229,))

In [21]:
# split into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

X_train.shape, X_test.shape

((46360, 18), (19869, 18))

In [22]:
# 'avg' is undefined (infinite) for the handful of players with 0 at-bats
# replace with missing, so it goes through the same imputation as everything else

X_train['avg'] = X_train['avg'].replace([np.inf, -np.inf], np.nan)
X_test['avg'] = X_test['avg'].replace([np.inf, -np.inf], np.nan)

X_train.isnull().mean()

g               0.000000
ab              0.000000
r               0.000000
h               0.000000
double          0.000000
triple          0.000000
hr              0.000000
rbi             0.000000
sb              0.000000
bb              0.000000
so              0.000000
avg             0.090121
weight          0.004659
height          0.004400
bats            0.007981
age             0.000345
experience      0.000000
team_win_pct    0.000000
dtype: float64

In [23]:
# missing value imputation: replace NA in numerical variables with the
# median calculated on the training set only

numerical_vars = ['ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'sb', 'bb', 'so',
                  'avg', 'weight', 'height', 'age', 'experience', 'team_win_pct']

for var in numerical_vars:

    value = X_train[var].median()

    X_train[var] = X_train[var].fillna(value)
    X_test[var] = X_test[var].fillna(value)

X_train[numerical_vars].isnull().sum()

ab              0
r               0
h               0
double          0
triple          0
hr              0
rbi             0
sb              0
bb              0
so              0
avg             0
weight          0
height          0
age             0
experience      0
team_win_pct    0
dtype: int64

In [24]:
# missing value imputation: replace NA in the categorical variable with the
# most frequent category, calculated on the training set only

value = X_train['bats'].mode()[0]

X_train['bats'] = X_train['bats'].fillna(value)
X_test['bats'] = X_test['bats'].fillna(value)

X_train['bats'].isnull().sum()

np.int64(0)

In [25]:
# outlier capping: find the IQR boundaries for a skewed count variable
# on the training set, then cap values beyond those boundaries

def find_skewed_boundaries(df, variable, distance):

    IQR = df[variable].quantile(0.75) - df[variable].quantile(0.25)

    lower_boundary = df[variable].quantile(0.25) - (IQR * distance)
    upper_boundary = df[variable].quantile(0.75) + (IQR * distance)

    return upper_boundary, lower_boundary


so_upper_limit, so_lower_limit = find_skewed_boundaries(X_train, 'so', 1.5)
so_upper_limit, so_lower_limit

(np.float64(96.5), np.float64(-51.5))

In [26]:
# replace the outliers by the maximum and minimum limit, in both train and test

X_train['so'] = np.where(X_train['so'] > so_upper_limit, so_upper_limit,
                         np.where(X_train['so'] < so_lower_limit, so_lower_limit, X_train['so']))

X_test['so'] = np.where(X_test['so'] > so_upper_limit, so_upper_limit,
                        np.where(X_test['so'] < so_lower_limit, so_lower_limit, X_test['so']))

In [27]:
# variable binning: not used as a model feature, just to explore the data -
# does the chance of being an All-Star change with age?

age_bins = pd.cut(X_train['age'], bins=[0, 24, 29, 34, 100],
                  labels=['young', 'prime', 'veteran', 'old'], include_lowest=True)

y_train.groupby(age_bins, observed=True).mean()

age
young      0.048021
prime      0.071473
veteran    0.087050
old        0.087926
Name: all_star, dtype: float64

In [28]:
# categorical encoding: one-hot encode 'bats' (batting hand: Left / Right / Both)

X_train_enc = pd.get_dummies(X_train, columns=['bats'], drop_first=True, dtype=int)
X_test_enc = pd.get_dummies(X_test, columns=['bats'], drop_first=True, dtype=int)

X_train_enc.head()

,g,ab,r,h,double,triple,hr,rbi,sb,bb,so,avg,weight,height,age,experience,team_win_pct,bats_L,bats_R
62949,39,59.0,5.0,19.0,4.0,0.0,0.0,8.0,0.0,1.0,4.0,0.322034,195.0,73.0,21.0,4,0.322581,1,0
54406,161,519.0,118.0,163.0,33.0,1.0,42.0,120.0,16.0,142.0,66.0,0.314066,190.0,71.0,28.0,8,0.493827,0,1
48415,98,312.0,32.0,68.0,16.0,1.0,2.0,22.0,4.0,24.0,57.0,0.217949,160.0,71.0,23.0,0,0.590062,0,1
7151,45,91.0,11.0,32.0,8.0,0.0,3.0,14.0,0.0,4.0,22.0,0.351648,195.0,71.0,25.0,1,0.401235,1,0
13638,35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.243009,185.0,75.0,29.0,6,0.428571,0,1


In [29]:
# feature scaling: standardise all variables using the parameters
# learned from the training set only

scaler = StandardScaler()
scaler.fit(X_train_enc)

X_train_scaled = scaler.transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)

### Step 4: Machine Learning Modelling

In [30]:
# naive baseline: always predict the majority class (not an All-Star)

majority_class = y_train.mode()[0]
naive_pred = pd.Series(majority_class, index=y_test.index)

naive_accuracy = (naive_pred == y_test).mean()
print('Naive baseline accuracy: {:.3f}'.format(naive_accuracy))

Naive baseline accuracy: 0.928


In [31]:
# simple machine learning model: Logistic Regression

model = LogisticRegression(random_state=0)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

model_accuracy = (y_pred == y_test).mean()
print('Logistic Regression accuracy: {:.3f}'.format(model_accuracy))

Logistic Regression accuracy: 0.933


In [32]:
# accuracy alone hides whether the model finds any real All-Stars at all -
# count how many true All-Stars in the test set were correctly identified

actual_all_stars = (y_test == 1).sum()
correctly_found = ((y_pred == 1) & (y_test == 1)).sum()
false_alarms = ((y_pred == 1) & (y_test == 0)).sum()

print('Actual All-Stars in test set:', actual_all_stars)
print('Correctly identified:', correctly_found)
print('False alarms (predicted All-Star, was not):', false_alarms)

Actual All-Stars in test set: 1421
Correctly identified: 234
False alarms (predicted All-Star, was not): 137


In [33]:
# compare the two approaches side by side

comparison = pd.DataFrame({
    'Model': ['Naive baseline', 'Logistic Regression'],
    'Accuracy': [naive_accuracy, model_accuracy],
    'All-Stars correctly found': [0, correctly_found],
})
comparison

,Model,Accuracy,All-Stars correctly found
0,Naive baseline,0.928482,0
1,Logistic Regression,0.933364,234


---
## Report

### 1. Table of Contents

1. Table of Contents
2. Introduction with Value Based Problem Statement
3. Problem Formulation
4. Data Wrangling on Multiple Tables
5. Data Cleansing and Transformation
6. Machine Learning Model
7. Summary and Further Improvements
8. Reflection

### 2. Introduction with Value Based Problem Statement

Being selected for Major League Baseball's All-Star Game is one of the highest honours a
player can receive in a season. It raises a player's profile and market value, giving clubs a
marketing and merchandising boost. Because voting happens mid-season and is partly subjective
(decided by fans, players and managers), an objective, data-driven estimate of a player's
All-Star chances is useful to clubs, agents and analysts. This project builds a simple model that
estimates whether a position player will be selected as an All-Star in a given season, based only
on statistics that can be assembled from the historical database.

### 3. Problem Formulation

After loading and exploring the `batting`, `player`, `team` and `all_star` tables, I formulated a
binary classification problem: for every position player-season, predict whether the player
(`all_star` = 1) was selected for that year's All-Star Game or not (`all_star` = 0). The population
was restricted to player-seasons with at least 20 games played, since the All-Star Game is played
mid-season and players with very little playing time realistically cannot be selected. Exploring
the resulting target showed that only around 1 in 10 player-seasons is a positive case, so the
classes are imbalanced, this shaped the choice to compare the model against a naive baseline in
step 4, rather than relying on accuracy alone.

### 4. Data Wrangling on Multiple Tables

The `batting` table has one row per player per stint (a player traded mid-season has more than
one row for that year), so rows were grouped by player and year and the counting statistics were
summed to get one row per player-season. Player-seasons with fewer than 20 games were filtered out and
the team each player played for was taken from the first matching row in `batting`. Batting average
was calculated as hits divided by at-bats (a mathematical calculation on the wrangled data). The
`player` table was merged in on `player_id` to bring in birth year, height, weight and batting hand.
The text/date column `debut` was parsed with `pd.to_datetime` to extract the debut year, from which
age and years of experience were calculated. Finally, the `team` table was merged in on year and
team to bring in that season's team win percentage, and the `all_star` table was merged in to build
the target variable, with players who were never listed treated as 0.

### 5. Data Cleansing and Transformation

The data was split into training and testing sets before any cleansing was fitted, so that every
step below learns its parameters from the training set only and is then applied unchanged to the
test set, avoiding data leakage. A handful of players had 0 at-bats, which made batting average
undefined (infinite), these were treated as missing values. Missing numerical values (batting
average, height, weight, etc.) were replaced with the training set median and the one missing
categorical value (batting hand) was replaced with the training set's most frequent category (mode).
Strikeouts were capped at the IQR-based outlier boundary instead of removing those rows, since a
few players with unusually high strikeout counts are not measurement errors. As an exploratory
step, age was binned into young / prime / veteran / old groups, which showed the All-Star rate
peaking in the "old" and "veteran" bands. Batting hand was one-hot encoded into numerical dummy columns, and all
features were standardised with `StandardScaler`, fitted on the training set only.

### 6. Machine Learning Model

A naive baseline that always predicts "not an All-Star" was built first, since it always achieves
a high accuracy purely because the classes are imbalanced. By definition, it never identifies a
single real All-Star. A Logistic Regression model was then trained on the standardised, cleansed
features. Its overall accuracy was similar to the naive baseline, which on its own would look
unimpressive, but unlike the naive baseline, the Logistic Regression model correctly identified
a number of real All-Stars in the test set, which the naive baseline can never do by construction.
This shows that even a simple linear model picked up a genuine signal from the wrangled features,
even though the class imbalance limits how many All-Stars it catches without further tuning.

### 7. Summary and Further Improvements

Starting from four relational tables, this project formulated a binary classification problem,
combined the tables into a single tabular dataset, cleansed and transformed the data, and compared
a simple Logistic Regression model against a naive baseline. The model's ability to correctly
identify real All-Stars, something the naive baseline cannot do at all, is the key evidence that
the wrangling and modelling added value. Further improvements could include: using a threshold
below 0.5 or class weighting to better handle the class imbalance, adding pitching and fielding
statistics for a more complete player profile and trying a Decision Tree model for comparison, to
see whether a non-linear model captures the pattern better than a linear one.

### 8. Reflection

This assignment showed me how much of the effort in a data science project goes into wrangling
before any modelling happens, combining four tables into one clean, well-defined dataset took far
longer than fitting the final model. I practised grouping and aggregating transactional data,
merging multiple tables on different keys, extracting a feature from a text/date column, and
building a full cleansing pipeline (missing values, outliers, encoding, scaling) that avoids
leakage by fitting everything on the training set only. The biggest lesson was realising how
misleading accuracy can be on an imbalanced target, and why comparing against a naive baseline
matters. If I did this again, I would explore class weighting earlier, since the imbalance limited
how many All-Stars the simple model could catch.